In [1]:
!pip install -q diffusers transformers xformers accelerate
!pip install -q numpy scipy ftfy Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00


In [2]:
import os
from typing import Optional, Tuple, List, Union

import torch
import numpy as np
from PIL import Image
from tqdm import trange

# HuggingFace Diffusers
from diffusers import (
    StableDiffusionPipeline,
    DDIMScheduler,
    AutoencoderKL,
    UNet2DConditionModel,
    ControlNetModel,
)

# HuggingFace Transformers
from transformers import (
    CLIPTextModel,
    CLIPTokenizer,
    CLIPModel,
    CLIPProcessor,
)

# Torchvision for data augmentation
from torchvision import transforms

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
def get_step_schedule(
    min_steps: int,
    max_steps: int,
    num_levels: int,
    schedule_type: str = 'linear'
) -> List[int]:
    """
    Generate step schedule for hierarchical interpolation.

    Args:
        min_steps: Starting step index
        max_steps: Ending step index
        num_levels: Number of hierarchy levels
        schedule_type: 'linear', 'convex', or 'concave'

    Returns:
        List of step indices [0, step1, step2, ..., stepN]

    Example:
        >>> get_step_schedule(50, 125, 3, 'linear')
        [0, 50, 87, 125]
    """
    diff = max_steps - min_steps

    if schedule_type == 'concave':
        return [0] + [
            int(diff * x**0.5) + min_steps
            for x in np.linspace(0, 1, num_levels)
        ]
    elif schedule_type == 'convex':
        return [0] + [
            int(diff * x**2) + min_steps
            for x in np.linspace(0, 1, num_levels)
        ]
    elif schedule_type == 'linear':
        return [0] + [
            int(x) for x in np.linspace(min_steps, max_steps, num_levels)
        ]
    else:
        raise ValueError(f"Unknown schedule_type: {schedule_type}")

In [4]:
@torch.no_grad()
def slerp(p0: torch.Tensor, p1: torch.Tensor, fract_mixing: float) -> torch.Tensor:
    """
    Spherical linear interpolation between two tensors.

    Preserves the norm of the interpolated vector, which is important
    for interpolating in latent spaces.

    Args:
        p0: First tensor
        p1: Second tensor
        fract_mixing: Mixing coefficient in [0, 1]
                     0 returns p0, 1 returns p1

    Returns:
        Interpolated tensor
    """
    # Determine dtype for recasting
    if p0.dtype == torch.float16:
        recast_to = 'fp16'
    else:
        recast_to = 'fp32'

    # Cast to highest precision available
    # MPS doesn't support float64, so use float32
    if p0.device.type == 'mps':
        # MPS: stay in float32
        p0 = p0.float()
        p1 = p1.float()
    else:
        # CUDA/CPU: use float64 for better numerical stability
        p0 = p0.double()
        p1 = p1.double()

    # Compute norms
    norm = torch.linalg.norm(p0) * torch.linalg.norm(p1)
    epsilon = 1e-7

    # Compute angle between vectors
    dot = torch.sum(p0 * p1) / norm
    dot = dot.clamp(-1 + epsilon, 1 - epsilon)

    theta_0 = torch.arccos(dot)
    sin_theta_0 = torch.sin(theta_0)

    # Interpolate
    theta_t = theta_0 * fract_mixing
    s0 = torch.sin(theta_0 - theta_t) / sin_theta_0
    s1 = torch.sin(theta_t) / sin_theta_0
    interp = p0 * s0 + p1 * s1

    # Recast to original dtype
    if recast_to == 'fp16':
        interp = interp.half()
    elif recast_to == 'fp32':
        interp = interp.float()
    # If we used double precision, convert back to float32
    elif interp.dtype == torch.float64:
        interp = interp.float()

    return interp

In [5]:
def interpolate_linear(p0: torch.Tensor, p1: torch.Tensor, frac: float) -> torch.Tensor:
    """
    Linear interpolation between two tensors.

    Args:
        p0: First tensor
        p1: Second tensor
        frac: Mixing coefficient in [0, 1]

    Returns:
        Interpolated tensor
    """
    return p0 + (p1 - p0) * frac

In [6]:
class DiffusersInterpolator:
    """
    Hierarchical bisection interpolation with quality control using HuggingFace Diffusers.

    Example:
        >>> interpolator = DiffusersInterpolator()
        >>> img1 = Image.open('apple.png')
        >>> img2 = Image.open('banana.png')
        >>> results = interpolator.interpolate_qc(img1, img2, num_frames=17)
    """

    def __init__(
        self,
        model_id: str = "runwayml/stable-diffusion-v1-5",
        controlnet_type: Optional[str] = None,
        device: Optional[str] = None,
        dtype: Optional[torch.dtype] = None,
    ):
        if device is None:
            if torch.cuda.is_available():
                device = "cuda"
            elif torch.backends.mps.is_available():
                device = "mps"
            else:
                device = "cpu"

        if dtype is None:
            if device == "cuda":
                dtype = torch.float16  # Use fp16 on CUDA for speed
            else:
                dtype = torch.float32  # MPS and CPU need fp32

        self.device = device
        self.dtype = dtype
        self.model_id = model_id

        print(f"Using device: {device} with dtype: {dtype}")

        print(f"Loading Stable Diffusion from {model_id}...")

        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
        self.vae.eval()

        self.unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype).to(device)
        self.unet.eval()

        self.text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
        self.text_encoder.eval()

        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")

        self.controlnet = None
        if controlnet_type:
            print(f"  Loading ControlNet ({controlnet_type})...")
            controlnet_id = self._get_controlnet_id(controlnet_type, model_id)
            self.controlnet = ControlNetModel.from_pretrained(
                controlnet_id,
                torch_dtype=dtype
            ).to(device)
            self.controlnet.eval()

        self.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")

        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
        self.clip_model.eval()

        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

        print("✓ All models loaded successfully!")

        self.vae_scale_factor = 2 ** (len(self.vae.config.block_out_channels) - 1)

    def _get_controlnet_id(self, controlnet_type: str, base_model: str) -> str:
        """
        Get HuggingFace ControlNet model ID based on type and base model.

        Args:
            controlnet_type: "canny", "openpose", "depth", "seg", etc.
            base_model: Base SD model ID

        Returns:
            ControlNet model ID
        """
        # Map controlnet types to HuggingFace model IDs
        if "stable-diffusion-v1-5" in base_model or "v1-5" in base_model:
            controlnet_map = {
                "canny": "lllyasviel/sd-controlnet-canny",
                "openpose": "lllyasviel/sd-controlnet-openpose",
                "depth": "lllyasviel/sd-controlnet-depth",
                "seg": "lllyasviel/sd-controlnet-seg",
                "mlsd": "lllyasviel/sd-controlnet-mlsd",
                "normal": "lllyasviel/sd-controlnet-normal",
                "scribble": "lllyasviel/sd-controlnet-scribble",
            }
        elif "stable-diffusion-2" in base_model:
            # SD 2.x ControlNets (fewer available)
            controlnet_map = {
                "canny": "thibaud/controlnet-sd21-canny-diffusers",
                "depth": "thibaud/controlnet-sd21-depth-diffusers",
                "openpose": "thibaud/controlnet-sd21-openpose-diffusers",
            }
        else:
            raise ValueError(f"Unknown base model: {base_model}")

        if controlnet_type not in controlnet_map:
            raise ValueError(
                f"Unknown controlnet type: {controlnet_type}. "
                f"Available: {list(controlnet_map.keys())}"
            )

        return controlnet_map[controlnet_type]

    @torch.no_grad()
    def _encode_prompt(self, prompt: str) -> torch.Tensor:
        """
        Encode text prompt to embeddings using CLIP.

        Args:
            prompt: Text prompt string

        Returns:
            text_embeddings: Tensor of shape (1, 77, 768) for SD1.5
                                          or (1, 77, 1024) for SD2.1
        """
        # Tokenize
        text_inputs = self.tokenizer(
            prompt,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )

        # Encode
        text_embeddings = self.text_encoder(
            text_inputs.input_ids.to(self.device)
        )[0]

        return text_embeddings

    @torch.no_grad()
    def _encode_image(self, image: Image.Image) -> torch.Tensor:
        """
        Encode PIL Image to latent representation.

        Args:
            image: PIL Image (RGB, should be 512x512)

        Returns:
            latent: Tensor of shape (1, 4, 64, 64)
        """
        # Convert PIL to tensor [0, 1]
        image_np = np.array(image).astype(np.float32) / 255.0

        # Normalize to [-1, 1]
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).unsqueeze(0)
        image_tensor = (image_tensor - 0.5) * 2.0
        image_tensor = image_tensor.to(device=self.device, dtype=self.dtype)

        # Encode to latent
        latent_dist = self.vae.encode(image_tensor).latent_dist
        latent = latent_dist.sample()

        # Scale (important for proper reconstruction)
        latent = latent * self.vae.config.scaling_factor

        return latent

    @torch.no_grad()
    def _decode_latent(self, latent: torch.Tensor) -> Image.Image:
        """
        Decode latent to PIL Image.

        Args:
            latent: Tensor of shape (1, 4, 64, 64)

        Returns:
            image: PIL Image (RGB)
        """
        # Unscale
        latent = latent / self.vae.config.scaling_factor

        # Decode
        image_tensor = self.vae.decode(latent).sample

        # Convert from [-1, 1] to [0, 1]
        image_tensor = (image_tensor / 2 + 0.5).clamp(0, 1)

        # Convert to numpy [0, 255]
        image_np = image_tensor.cpu().permute(0, 2, 3, 1).float().numpy()
        image_np = (image_np * 255).round().astype(np.uint8)

        # Convert to PIL
        image = Image.fromarray(image_np[0])

        return image

    def _get_latent_stack(
        self,
        img1: Image.Image,
        img2: Image.Image,
        timestep_schedule: List[int],
        share_noise: bool = True
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """
        Build progressively noisier latents with shared noise schedule.

        This is critical for hierarchical interpolation: we add noise incrementally
        using the SAME noise realization for both images, ensuring interpolated
        latents denoise properly.

        Args:
            img1, img2: Input images
            timestep_schedule: List of timesteps [0, 250, 435, 625, ...]
                              0 = clean, higher = more noise
            share_noise: Use same noise for both images (True for temporal coherence)

        Returns:
            (latents1, latents2): Lists of noisy latents at each timestep level
                                 latents1[0] = clean L1, latents1[-1] = max noise
        """
        # Encode clean images
        L1 = self._encode_image(img1)
        L2 = self._encode_image(img2)

        latents1 = [L1]
        latents2 = [L2]

        # Progressively add noise
        for i in range(1, len(timestep_schedule)):
            t_prev = timestep_schedule[i - 1] if i > 0 else None
            t_now = timestep_schedule[i]

            # Generate noise
            noise = torch.randn_like(L1)

            # Add noise to first image
            latents1.append(
                self._add_noise(latents1[-1], noise, t_now, t_prev)
            )

            # Add noise to second image (same noise if share_noise=True)
            if not share_noise:
                noise = torch.randn_like(L2)

            latents2.append(
                self._add_noise(latents2[-1], noise, t_now, t_prev)
            )

        return latents1, latents2

    def _add_noise(
        self,
        latent: torch.Tensor,
        noise: torch.Tensor,
        t_now: int,
        t_prev: Optional[int] = None
    ) -> torch.Tensor:
        """
        Add noise to latent using DDIM forward process.

        If t_prev is None: Add noise from scratch (x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε)
        If t_prev is given: Add incremental noise from t_prev to t_now

        Args:
            latent: Current latent (clean or partially noised)
            noise: Noise to add
            t_now: Target timestep
            t_prev: Previous timestep (None if starting from clean)

        Returns:
            Noisy latent at timestep t_now
        """
        # Get alpha values from scheduler
        alphas_cumprod = self.scheduler.alphas_cumprod.to(self.device)

        if t_prev is None:
            # Add noise from scratch: x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε
            sqrt_alpha = alphas_cumprod[t_now] ** 0.5
            sqrt_one_minus_alpha = (1 - alphas_cumprod[t_now]) ** 0.5

            noisy_latent = sqrt_alpha * latent + sqrt_one_minus_alpha * noise

        else:
            # Add incremental noise from t_prev to t_now
            # Given: x_{t_prev} = √ᾱ_{t_prev} * x_0 + √(1-ᾱ_{t_prev}) * ε_{prev}
            # Want:  x_{t_now} = √ᾱ_{t_now} * x_0 + √(1-ᾱ_{t_now}) * ε_{combined}
            # Where ε_{combined} includes ε_{prev} plus new noise

            a_prev = alphas_cumprod[t_prev] ** 0.5
            a_now = alphas_cumprod[t_now] ** 0.5
            sig_prev = (1 - alphas_cumprod[t_prev]) ** 0.5
            sig_now = (1 - alphas_cumprod[t_now]) ** 0.5

            # Scale factor for existing latent
            scale = a_now / a_prev

            # Additional noise needed
            sigma = (sig_now**2 - (scale * sig_prev)**2) ** 0.5

            noisy_latent = scale * latent + sigma * noise

        return noisy_latent

    def _denoise_step(
        self,
        latent: torch.Tensor,
        text_embeddings: torch.Tensor,
        timestep: int,
        guidance_scale: float = 7.5,
        controlnet_image: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Single DDIM denoising step.

        Uses classifier-free guidance: noise_pred = uncond + scale * (cond - uncond)

        Args:
            latent: Current noisy latent (1, 4, 64, 64)
            text_embeddings: Concatenated [uncond, cond] embeddings (2, 77, 768)
            timestep: Current timestep (int, e.g., 500)
            guidance_scale: CFG strength (typically 7.5)
            controlnet_image: Optional ControlNet conditioning image

        Returns:
            Denoised latent (one step less noise)
        """
        # Prepare timestep tensor
        t = torch.tensor([timestep], device=self.device, dtype=torch.long)

        # Duplicate latent for CFG (uncond + cond)
        latent_model_input = torch.cat([latent, latent])

        # Scale latent for UNet (some schedulers require this)
        latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)

        # ControlNet forward pass (if enabled)
        down_block_res_samples = None
        mid_block_res_sample = None

        if self.controlnet is not None and controlnet_image is not None:
            # Duplicate control image for CFG
            controlnet_cond = torch.cat([controlnet_image, controlnet_image])

            down_block_res_samples, mid_block_res_sample = self.controlnet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings,
                controlnet_cond=controlnet_cond,
                return_dict=False,
            )

        # UNet forward pass
        noise_pred = self.unet(
            latent_model_input,
            t,
            encoder_hidden_states=text_embeddings,
            down_block_additional_residuals=down_block_res_samples,
            mid_block_additional_residual=mid_block_res_sample,
        ).sample

        # Perform classifier-free guidance
        noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (
            noise_pred_cond - noise_pred_uncond
        )

        # Scheduler step (denoise one step)
        latent = self.scheduler.step(
            noise_pred,
            timestep,
            latent,
            return_dict=False
        )[0]

        # Clear MPS cache to prevent memory accumulation
        if self.device == 'mps':
            torch.mps.empty_cache()

        return latent

    def _optimize_embeddings(
        self,
        img1: Image.Image,
        img2: Image.Image,
        prompt: str,
        n_prompt: str,
        num_iters: int = 200,
        lr: float = 1e-4,
        guide_scale: float = 7.5
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Optimize text embeddings to reconstruct images (textual inversion).

        Instead of using generic prompts, finds custom embeddings that help
        the model reconstruct each specific image.

        Args:
            img1, img2: Input images
            prompt: Base text prompt
            n_prompt: Negative prompt
            num_iters: Optimization iterations (200-500 recommended)
            lr: Learning rate (1e-4 default)
            guide_scale: CFG strength during optimization

        Returns:
            (cond1, cond2, uncond): Optimized embeddings for img1, img2, and negative
        """
        print(f"Optimizing text embeddings ({num_iters} iterations)...")

        # Data augmentation (prevents overfitting)
        augment = transforms.Compose([
            transforms.ColorJitter(
                brightness=0.1, contrast=0.2, saturation=0.2, hue=0.1
            ),
            transforms.RandomResizedCrop(size=(512, 512), scale=(0.7, 1.0)),
        ])

        # Get base embeddings
        with torch.no_grad():
            cond_base = self._encode_prompt(prompt)
            uncond_base = self._encode_prompt(n_prompt)

        # Create learnable copies
        cond1 = cond_base.clone().detach().requires_grad_(True)
        cond2 = cond_base.clone().detach().requires_grad_(True)
        uncond = uncond_base.clone().detach().requires_grad_(True)

        # Optimizer
        optimizer = torch.optim.Adam([cond1, cond2, uncond], lr=lr)

        # Setup scheduler for optimization
        self.scheduler.set_timesteps(50)  # Dummy schedule to get timesteps
        timesteps = self.scheduler.timesteps.cpu().numpy()

        # Training loop
        for iteration in range(num_iters):
            # Random timestep from middle range (not too easy, not too hard)
            t_idx = np.random.randint(len(timesteps) // 3, 2 * len(timesteps) // 3)
            t = int(timesteps[t_idx])

            # === Optimize embedding for img1 ===

            # Apply augmentation
            img1_aug = augment(img1)

            # Encode augmented image
            with torch.no_grad():
                L1 = self._encode_image(img1_aug)

            # Add noise at random timestep
            noise = torch.randn_like(L1)
            alphas_cumprod = self.scheduler.alphas_cumprod.to(self.device)
            sqrt_alpha = alphas_cumprod[t] ** 0.5
            sqrt_one_minus_alpha = (1 - alphas_cumprod[t]) ** 0.5
            noisy_L1 = sqrt_alpha * L1 + sqrt_one_minus_alpha * noise

            # Predict noise using current cond1
            text_emb = torch.cat([uncond, cond1])
            latent_input = torch.cat([noisy_L1, noisy_L1])
            t_tensor = torch.tensor([t] * 2, device=self.device, dtype=torch.long)

            # UNet forward pass
            noise_pred = self.unet(
                latent_input,
                t_tensor,
                encoder_hidden_states=text_emb,
            ).sample

            # Apply CFG
            noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
            noise_pred_guided = noise_pred_uncond + guide_scale * (
                noise_pred_cond - noise_pred_uncond
            )

            # Loss: MSE between predicted and true noise
            loss1 = torch.nn.functional.mse_loss(noise_pred_guided, noise)

            # === Optimize embedding for img2 ===

            img2_aug = augment(img2)

            with torch.no_grad():
                L2 = self._encode_image(img2_aug)

            noise = torch.randn_like(L2)
            noisy_L2 = sqrt_alpha * L2 + sqrt_one_minus_alpha * noise

            text_emb = torch.cat([uncond, cond2])
            latent_input = torch.cat([noisy_L2, noisy_L2])

            noise_pred = self.unet(
                latent_input,
                t_tensor,
                encoder_hidden_states=text_emb,
            ).sample

            noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
            noise_pred_guided = noise_pred_uncond + guide_scale * (
                noise_pred_cond - noise_pred_uncond
            )

            loss2 = torch.nn.functional.mse_loss(noise_pred_guided, noise)

            # Backprop and update
            total_loss = loss1 + loss2
            total_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Log progress
            if iteration % 50 == 0:
                print(f"  Iter {iteration}/{num_iters}: "
                      f"loss1={loss1.item():.4f}, loss2={loss2.item():.4f}")

        print(f"✓ Optimization complete!")

        return cond1.detach(), cond2.detach(), uncond.detach()

    @torch.no_grad()
    def _evaluate_with_clip(
        self,
        image: Image.Image,
        pos_prompt: str,
        neg_prompt: str
    ) -> float:
        """
        Score image using CLIP similarity.

        Args:
            image: PIL Image to evaluate
            pos_prompt: Positive text (e.g., "high quality, detailed")
            neg_prompt: Negative text (e.g., "blurry, distorted")

        Returns:
            score: Float (higher is better)
                  score = similarity(image, pos) - similarity(image, neg)
        """
        # Process inputs
        inputs = self.clip_processor(
            text=[pos_prompt, neg_prompt],
            images=image,
            return_tensors="pt",
            padding=True
        )

        # Move to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Get embeddings
        outputs = self.clip_model(**inputs)
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

        # Compute similarities
        pos_sim = torch.nn.functional.cosine_similarity(
            image_embeds, text_embeds[0:1]
        )
        neg_sim = torch.nn.functional.cosine_similarity(
            image_embeds, text_embeds[1:2]
        )

        # Score = positive similarity - negative similarity
        score = (pos_sim - neg_sim).item()

        return score

    def interpolate_qc(
        self,
        img1: Image.Image,
        img2: Image.Image,
        prompt: str = "a photograph",
        n_prompt: str = "low quality, blurry",
        qc_prompts: Optional[Tuple[str, str]] = None,
        num_frames: int = 17,
        n_choices: int = 8,
        min_steps: float = 0.3,
        max_steps: float = 0.55,
        ddim_steps: int = 250,
        guide_scale: float = 7.5,
        optimize_cond: int = 0,
        cond_lr: float = 1e-4,
        latent_interp: str = 'slerp',
        schedule_type: str = 'linear',
        out_dir: str = 'output',
        seed: Optional[int] = None
    ) -> List[Image.Image]:
        """
        Hierarchical bisection interpolation with CLIP-based quality control.

        Args:
            img1, img2: Input images (should be 512x512 RGB)
            prompt: Text prompt for generation
            n_prompt: Negative prompt
            qc_prompts: (positive, negative) for CLIP scoring
                       If None, uses manual selection
            num_frames: Total frames (num_frames-1 must be power of 2)
            n_choices: Candidates generated per frame
            min_steps, max_steps: Noise range (0-1 or absolute indices)
            ddim_steps: Total DDIM timesteps
            guide_scale: Classifier-free guidance strength
            optimize_cond: Textual inversion iterations (0 to disable)
            cond_lr: Learning rate for textual inversion
            latent_interp: 'slerp' or 'linear'
            schedule_type: 'linear', 'convex', or 'concave'
            out_dir: Output directory
            seed: Random seed for reproducibility

        Returns:
            List of PIL Images (interpolation sequence)
        """
        # Set random seed
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)

        # Setup output directory
        os.makedirs(out_dir, exist_ok=True)
        img1.save(f'{out_dir}/000.png')
        img2.save(f'{out_dir}/{num_frames-1:03d}.png')

        # Validate num_frames
        num_levels = int(np.log2(num_frames - 1))
        assert 2**num_levels == num_frames - 1, \
            f"num_frames-1 must be power of 2, got {num_frames-1}"

        print(f"\n{'='*70}")
        print(f"Starting Hierarchical Interpolation")
        print(f"{'='*70}")
        print(f"Frames: {num_frames} ({num_levels} levels)")
        print(f"DDIM steps: {ddim_steps}")
        print(f"Candidates per frame: {n_choices}")
        print(f"Quality control: {'CLIP' if qc_prompts else 'Manual'}")
        print(f"Textual inversion: {optimize_cond} iters" if optimize_cond else "Textual inversion: Disabled")
        print(f"{'='*70}\n")

        # Convert min/max_steps to absolute if needed
        if min_steps < 1:
            min_steps = int(ddim_steps * min_steps)
        if max_steps < 1:
            max_steps = int(ddim_steps * max_steps)

        # Setup DDIM scheduler
        self.scheduler.set_timesteps(ddim_steps)
        timesteps = self.scheduler.timesteps.cpu().numpy()

        # Get step schedule for hierarchy
        step_schedule = get_step_schedule(
            int(min_steps), int(max_steps), num_levels, schedule_type
        )

        print(f"Step schedule: {step_schedule}")
        print(f"Timestep range: {timesteps[step_schedule[-1]]} → {timesteps[step_schedule[1]]}\n")

        # Textual inversion (optional)
        if optimize_cond > 0:
            cond1, cond2, uncond = self._optimize_embeddings(
                img1, img2, prompt, n_prompt,
                num_iters=optimize_cond, lr=cond_lr, guide_scale=guide_scale
            )
            print()
        else:
            print("Using base text embeddings (no optimization)...")
            cond1 = self._encode_prompt(prompt)
            cond2 = cond1.clone()
            uncond = self._encode_prompt(n_prompt)
            print()

        # Encode endpoint images
        print("Encoding endpoint images...")
        final_latents = [None] * num_frames
        final_latents[0] = self._encode_image(img1)
        final_latents[-1] = self._encode_image(img2)
        print(f"Latent shape: {final_latents[0].shape}\n")

        # Choose interpolation function
        interp_fn = slerp if latent_interp == 'slerp' else interpolate_linear

        # Hierarchical generation
        print(f"{'='*70}")
        print("Hierarchical Generation")
        print(f"{'='*70}\n")

        for level in range(1, num_levels + 1):
            cur_step = step_schedule[-level]
            t = int(timesteps[cur_step])
            df = 2 ** (num_levels - level)  # Frame step at this level

            print(f"Level {level}/{num_levels}:")
            print(f"  Timestep: {t}")
            print(f"  Frame step: {df}")
            print(f"  Frames to generate: {len(range(df, num_frames-1, df*2))}")

            for frame_ix in range(df, num_frames - 1, df * 2):
                frac = frame_ix / (num_frames - 1)

                print(f"    Generating frame {frame_ix}/{num_frames-1} (α={frac:.3f})...")

                # Interpolate text conditioning
                cond_interp = interp_fn(cond1, cond2, frac)
                text_emb = torch.cat([uncond, cond_interp])

                # Generate N candidates
                candidates = []
                clip_scores = []

                for choice_ix in range(n_choices):
                    # Add noise to neighboring final_latents
                    noise = torch.randn_like(final_latents[0])

                    alphas_cumprod = self.scheduler.alphas_cumprod.to(self.device)
                    sqrt_alpha = (alphas_cumprod[t] ** 0.5).item()
                    sqrt_one_minus = ((1 - alphas_cumprod[t]) ** 0.5).item()

                    # Noise the neighbors
                    l1 = sqrt_alpha * final_latents[frame_ix - df] + sqrt_one_minus * noise
                    l2 = sqrt_alpha * final_latents[frame_ix + df] + sqrt_one_minus * noise

                    # Interpolate
                    noisy_latent = interp_fn(l1, l2, 0.5)

                    # Denoise from cur_step to 0
                    for step_idx in range(cur_step, -1, -1):
                        t_cur = int(timesteps[step_idx])
                        noisy_latent = self._denoise_step(
                            noisy_latent, text_emb, t_cur, guide_scale
                        )

                    candidates.append(noisy_latent)

                    # Decode and evaluate
                    image = self._decode_latent(noisy_latent)

                    if qc_prompts:
                        # Automatic CLIP scoring
                        score = self._evaluate_with_clip(
                            image, qc_prompts[0], qc_prompts[1]
                        )
                        clip_scores.append(score)
                    else:
                        # Manual selection - save all candidates
                        image.save(f'{out_dir}/{frame_ix:03d}_{choice_ix}.png')

                # Select best candidate
                if qc_prompts:
                    best_idx = int(np.argmax(clip_scores))
                    print(f"      Selected candidate {best_idx} "
                          f"(score: {clip_scores[best_idx]:.4f})")

                    image = self._decode_latent(candidates[best_idx])
                    image.save(f'{out_dir}/{frame_ix:03d}.png')

                else:
                    # Manual selection
                    print(f"      Saved {n_choices} candidates. Choose 0-{n_choices-1}:")
                    choice = int(input("      Choice: "))
                    best_idx = choice

                    # Clean up non-selected candidates
                    for i in range(n_choices):
                        if i != choice:
                            os.remove(f'{out_dir}/{frame_ix:03d}_{i}.png')
                        else:
                            os.rename(
                                f'{out_dir}/{frame_ix:03d}_{i}.png',
                                f'{out_dir}/{frame_ix:03d}.png'
                            )

                # Cache the selected latent
                final_latents[frame_ix] = candidates[best_idx]

            # Reduce choices at finer levels
            n_choices = max(n_choices - 1, 3)

            print()

        # Decode all final frames
        print(f"{'='*70}")
        print("Decoding Final Sequence")
        print(f"{'='*70}\n")

        result_images = []
        for i in trange(num_frames, desc="Decoding frames"):
            if i == 0:
                result_images.append(img1)
            elif i == num_frames - 1:
                result_images.append(img2)
            else:
                # Check if already saved
                if os.path.exists(f'{out_dir}/{i:03d}.png'):
                    img = Image.open(f'{out_dir}/{i:03d}.png')
                else:
                    img = self._decode_latent(final_latents[i])
                    img.save(f'{out_dir}/{i:03d}.png')
                result_images.append(img)

        print(f"\n✓ Interpolation complete!")
        print(f"✓ {num_frames} frames saved to {out_dir}/")

        return result_images

In [7]:
print("Initializing interpolator...")
interpolator = DiffusersInterpolator(
    model_id="runwayml/stable-diffusion-v1-5"
)

# Load images (replace with your own images)
print("\nLoading images...")
img1 = Image.open('/noface1.png').convert('RGB').resize((512, 512))
img2 = Image.open('/noface2.jpeg').convert('RGB').resize((512, 512))

# Run interpolation
print("\nRunning interpolation...")
results = interpolator.interpolate_qc(
    img1, img2,
    prompt="a photograph of a person",
    n_prompt="low quality, blurry, distorted",
    qc_prompts=("high quality photograph", "blurry, low quality"),  # Automatic CLIP selection
    num_frames=9,           # 9 frames (8 intervals = 2^3)
    n_choices=3,            # Generate 3 candidates per frame
    ddim_steps=50,          # Faster (use 200 for quality)
    min_steps=0.3,          # 30% noise minimum
    max_steps=0.5,          # 50% noise maximum
    optimize_cond=0,        # Skip textual inversion for speed (use 200 for quality)
    latent_interp='slerp',  # Spherical interpolation
    out_dir='output_basic',
    seed=42                 # Reproducibility
)

print(f"\n✓ Done! Generated {len(results)} frames in output_basic/")

# Create a contact sheet
from PIL import ImageDraw, ImageFont

print("\nCreating contact sheet...")
thumb_size = 128
contact_sheet = Image.new('RGB', (thumb_size * len(results), thumb_size))

for i, img in enumerate(results):
    thumb = img.resize((thumb_size, thumb_size))
    contact_sheet.paste(thumb, (i * thumb_size, 0))

contact_sheet.save('content/output_basic/contact_sheet.png')
print("✓ Saved contact sheet to output_basic/contact_sheet.png")

Initializing interpolator...
Using device: cuda with dtype: torch.float16
Loading Stable Diffusion from runwayml/stable-diffusion-v1-5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✓ All models loaded successfully!

Loading images...

Running interpolation...

Starting Hierarchical Interpolation
Frames: 9 (3 levels)
DDIM steps: 50
Candidates per frame: 3
Quality control: CLIP
Textual inversion: Disabled

Step schedule: [0, 15, 20, 25]
Timestep range: 481 → 681

Using base text embeddings (no optimization)...

Encoding endpoint images...
Latent shape: torch.Size([1, 4, 64, 64])

Hierarchical Generation

Level 1/3:
  Timestep: 481
  Frame step: 4
  Frames to generate: 1
    Generating frame 4/8 (α=0.500)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 4.81 MiB is free. Including non-PyTorch memory, this process has 79.23 GiB memory in use. Of the allocated memory 78.72 GiB is allocated by PyTorch, and 9.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)